# Notebook 7: Temporal Dynamics of Cell Populations

Longitudinal analysis of cell counts, proportions, and spatial densities across treatment timepoints (Pre → JustAfter CRT → Resection).

**Overview:**
1. Major cluster composition — stacked bar plots (absolute count & proportion) per sample
2. Timepoint-level violin plots with statistical annotations (Mann-Whitney, BH correction)
3. Per-subcluster temporal dynamics for each cell type (Tumor, T cell, Myeloid, Stromal/CAF):
   - **Count**: raw cell count per sample
   - **Density**: cells per µm² of tissue (normalised by tissue area)
   - **Proportion**: fraction of total cells (normalised by total cell count)
4. UMAP embedding density for tumour cells stratified by Timepoint × Response

**Input files:**
- `../data/bicrc_integrated_annotated_.h5ad`
- `../data/integrate_adata_filtered_tumor.h5ad`
- `../data/cell_area.txt` — tissue area (µm²) per sample (index: Sample, column: cell_area)
- `../data/cell_count.txt` — total cell count per sample (index: Sample, column: cell_area)


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import scanpy as sc
import itertools
from scipy.stats import ttest_rel, ttest_ind, mannwhitneyu
from statsmodels.stats.multitest import multipletests
from statannotations.Annotator import Annotator

sc.settings.verbosity = 1
sc.settings.figdir = '../figures/'
plt.rcParams['figure.dpi'] = 150


## 2. Cohort Configuration

In [ ]:
# ── Sample lists ──────────────────────────────────────────────────────
sample_name_ls = [
    'Pt-1_Pre',  'Pt-1_Resection',
    'Pt-2_Pre',  'Pt-2_Resection',
    'Pt-3_Pre',  'Pt-3_JustAfter',  'Pt-3_Resection',
    'Pt-4_Pre',  'Pt-4_Resection',
    'Pt-5_Pre',  'Pt-5_JustAfter',  'Pt-5_Resection',
    'Pt-6_Pre',  'Pt-6_Resection',
    'Pt-7_Pre',  'Pt-7_Resection',
    'Pt-8_Pre',  'Pt-8_Resection',
    'Pt-9_Pre',  'Pt-9_Resection',
    'Pt-10_Pre', 'Pt-10_Resection',
    'Pt-11_Pre', 'Pt-11_JustAfter', 'Pt-11_Resection',
    'Pt-12_Pre', 'Pt-12_Resection',
    'Pt-13_Pre', 'Pt-13_JustAfter', 'Pt-13_Resection',
    'Pt-14_Pre', 'Pt-14_Resection',
    'Pt-15_Pre', 'Pt-15_Resection',
    'Pt-16_Pre', 'Pt-16_Resection',
    'Pt-17_Pre', 'Pt-17_Resection',
    'Pt-18_Pre', 'Pt-18_Resection',
    'Pt-19_Pre', 'Pt-19_Resection',
    'Pt-20_Pre', 'Pt-20_Resection',
    'Pt-21_Pre', 'Pt-21_Resection',
    'Pt-22_Pre', 'Pt-22_Resection',
    'Pt-23_Pre', 'Pt-23_Resection',
    'Pt-24_Pre', 'Pt-24_Resection',
]

# Patient ordering per response group (sorted by DFS status to aid visual inspection)
# Marker: 'o' = DFS (disease-free), 'x' = PD (progressive disease)
RESPONSE_GROUPS = {
    'NR': {
        'patients': ['Pt-3', 'Pt-1', 'Pt-2', 'Pt-4', 'Pt-5', 'Pt-6', 'Pt-7', 'Pt-8'],
        'dfs':      ['DFS',  'PD',   'PD',   'PD',   'PD',   'PD',   'PD',   'PD'],
        'color': 'red',
    },
    'MPR': {
        'patients': ['Pt-11', 'Pt-9',  'Pt-10', 'Pt-12', 'Pt-13', 'Pt-14', 'Pt-15', 'Pt-16'],
        'dfs':      ['DFS',   'PD',    'PD',    'DFS',   'PD',    'PD',    'DFS',   'PD'],
        'color': 'orange',
    },
    'pCR': {
        'patients': ['Pt-17', 'Pt-18', 'Pt-19', 'Pt-20', 'Pt-21', 'Pt-22', 'Pt-23', 'Pt-24'],
        'dfs':      ['DFS'] * 8,
        'color': 'blue',
    },
}

TIMEPOINT_ORDER = ['Pre', 'JustAfter', 'Resection']
CELL_TYPE_LS    = ['Tumor', 'Normal', 'Stromal', 'Myeloid', 'T cell', 'B cell']

CELL_TYPE_COLORS = {
    'Tumor':   '#F4B183',
    'Normal':  '#9DC3E6',
    'Stromal': sc.pl.palettes.default_20[1],
    'Myeloid': sc.pl.palettes.default_20[2],
    'T cell':  sc.pl.palettes.default_20[3],
    'B cell':  sc.pl.palettes.default_20[4],
}
# Tumor and Normal are shown with hatching (open interior) to indicate pathological distinction
HATCH_TYPES = {'Tumor': '///', 'Normal': '///'}

pre_samples        = [f'Pt-{i}_Pre'       for i in range(1, 25)]
just_after_samples = ['Pt-3_JustAfter', 'Pt-5_JustAfter', 'Pt-11_JustAfter', 'Pt-13_JustAfter']
resection_samples  = [f'Pt-{i}_Resection' for i in range(1, 25)]


## 3. Load Data

In [ ]:
adata = sc.read_h5ad('../data/bicrc_integrated_annotated_.h5ad')
adata_tumor = sc.read_h5ad('../data/integrate_adata_filtered_tumor.h5ad')

# Tissue area (µm²) and total cell count per sample
df_cellarea  = pd.read_table('../data/cell_area.txt',  index_col='Sample')
df_cellcount = pd.read_table('../data/cell_count.txt', index_col='Sample')

# Attach response / timepoint metadata to major obs
adata.obs['Patient']   = adata.obs['Sample'].str.split('_').str[0]
adata.obs['Timepoint'] = adata.obs['Sample'].str.split('_').str[1]

def sample_to_response(sample):
    pt = int(sample.split('_')[0].replace('Pt-', ''))
    if pt <= 8:  return 'NR'
    if pt <= 16: return 'MPR'
    return 'pCR'

adata.obs['Response'] = adata.obs['Sample'].apply(sample_to_response)
print(adata)


## 4. Major Cluster Composition

Stacked bar plots of **absolute cell counts** (top) and **cell proportions** (bottom) per sample, grouped by timepoint. Tumour and Normal epithelial bars are shown with hatching to indicate their pathological origin.

Tick labels are coloured **red** (progressive disease) or **black** (disease-free survival).

In [ ]:
# Build count matrix: index = Sample, columns = cell type
df_count = pd.DataFrame(index=sample_name_ls)
for ct in CELL_TYPE_LS:
    df_count[ct] = (
        adata.obs[adata.obs['Major_cluster_pathol'] == ct]['Sample']
        .value_counts()
        .reindex(sample_name_ls, fill_value=0)
    )
df_count.fillna(0, inplace=True)

# Annotate
df_count['Patient']   = [s.split('_')[0] for s in df_count.index]
df_count['Timepoint'] = [s.split('_', 1)[1] for s in df_count.index]
df_count['Response']  = df_count.index.map(sample_to_response)
df_count['DFS'] = df_count['Patient'].map({
    **{f'Pt-{i}': 'DFS' if i in {3,11,12,15,17,18,19,20,21,22,23,24} else 'PD'
       for i in range(1, 25)}
})

relapse_tick_color = {'DFS': 'black', 'PD': 'red'}

def stacked_bar(ax, sample_list, df, normalize=False, ylabel='', title=''):
    """Draw a stacked bar chart for the given sample_list."""
    plot_df = df.loc[[s for s in sample_list if s in df.index], CELL_TYPE_LS].fillna(0)
    if normalize:
        row_sums = plot_df.sum(axis=1).replace(0, np.nan)
        plot_df = plot_df.div(row_sums, axis=0).fillna(0)

    x = np.arange(len(plot_df))
    bottom = np.zeros(len(plot_df))

    for ct in CELL_TYPE_LS:
        vals = plot_df[ct].values
        hatch = HATCH_TYPES.get(ct)
        ax.bar(
            x, vals, 0.85, bottom=bottom,
            color='white' if hatch else CELL_TYPE_COLORS[ct],
            edgecolor=CELL_TYPE_COLORS[ct],
            hatch=hatch,
            label=ct,
        )
        bottom += vals

    ax.set_title(title)
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df.index, rotation=90, fontsize=7)
    for tick, sample in zip(ax.get_xticklabels(), plot_df.index):
        status = df.loc[sample, 'DFS'] if sample in df.index else 'PD'
        tick.set_color(relapse_tick_color.get(status, 'black'))
    ax.set_ylabel(ylabel)
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(False)

for normalize, ylabel, suffix in [(False, 'Cell count', 'count'), (True, 'Proportion', 'prop')]:
    fig, axes = plt.subplots(
        1, 3, figsize=(16, 5), sharey=True,
        gridspec_kw=dict(width_ratios=[len(pre_samples), len(just_after_samples), len(resection_samples)])
    )
    stacked_bar(axes[0], pre_samples,        df_count, normalize, ylabel, 'Pre')
    stacked_bar(axes[1], just_after_samples, df_count, normalize, '',     'JustAfter')
    stacked_bar(axes[2], resection_samples,  df_count, normalize, '',     'Resection')

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.05), ncol=6, frameon=False)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(f'../figures/07_major_cluster_{suffix}.png', dpi=300, bbox_inches='tight')
    plt.show()


## 5. Violin Plots by Timepoint

For each major cell type, compare absolute counts across Pre / JustAfter / Resection using Mann-Whitney U tests with Benjamini-Hochberg correction.

In [ ]:
df_violin = df_count.copy()
df_violin['Timepoint'] = pd.Categorical(df_violin['Timepoint'], categories=TIMEPOINT_ORDER, ordered=True)
timepoint_palette = {'Pre': 'red', 'JustAfter': 'gold', 'Resection': 'royalblue'}

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
axes = axes.flat

for ax, ct in zip(axes, CELL_TYPE_LS):
    sns.violinplot(
        data=df_violin, x='Timepoint', y=ct,
        order=TIMEPOINT_ORDER, palette=timepoint_palette,
        alpha=0.6, inner='quartile', cut=0, ax=ax
    )
    # Statistical annotation (all pairwise, BH corrected)
    pairs = list(itertools.combinations(TIMEPOINT_ORDER, 2))
    annotator = Annotator(ax, pairs, data=df_violin, x='Timepoint', y=ct, order=TIMEPOINT_ORDER)
    annotator.configure(
        test='Mann-Whitney',
        text_format='star',
        loc='outside',
        comparisons_correction='BH',
        verbose=False,
    )
    annotator.apply_and_annotate()
    ax.set_title(ct)
    ax.set_xlabel('')
    ax.set_ylabel('Cell count')
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Major Cell Type Count by Timepoint', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../figures/07_violin_major_cluster_timepoint.png', dpi=300, bbox_inches='tight')
plt.show()


## 6. Subcluster Temporal Dynamics

For each cell type (Tumour, T cell, Myeloid, Stromal/CAF), three DataFrames are computed:

| DataFrame | Description |
|-----------|-------------|
| `df_count` | Raw subcluster cell count per sample |
| `df_density` | Cells per µm² of tissue area (from `cell_area.txt`) |
| `df_prop` | Fraction of all cells in the sample (from `cell_count.txt`) |

Line plots show each patient's trajectory across timepoints. Marker style indicates prognosis: **●** = DFS, **✕** = PD. Statistical annotations compare Pre vs Resection (paired t-test within response) and cross-group at each timepoint (Welch's t-test).

In [ ]:
def build_metric_frames(adata_sub, sample_order, groupby='subcluster'):
    """Compute count / density / proportion DataFrames for subcluster temporal analysis.

    Parameters
    ----------
    adata_sub : AnnData subset for a single cell type
    sample_order : list of sample names defining the row order
    groupby : obs column used for subcluster labels

    Returns
    -------
    df_count, df_density, df_prop — each indexed by sample_order with metadata columns
    """
    # Raw crosstab (samples × subclusters)
    df_c = pd.crosstab(adata_sub.obs['Sample'], adata_sub.obs[groupby])
    df_c = pd.DataFrame(index=sample_order).join(df_c, how='left').fillna(0)

    subcluster_cols = [c for c in df_c.columns]

    def add_meta(df):
        df = df.copy()
        df['Timepoint'] = pd.Categorical(
            [s.split('_', 1)[1] for s in df.index],
            categories=TIMEPOINT_ORDER, ordered=True
        )
        df['Response'] = [sample_to_response(s) for s in df.index]
        df['Patient']  = [s.split('_')[0] for s in df.index]
        return df

    df_count   = add_meta(df_c)
    df_density = add_meta((df_c[subcluster_cols].T / df_cellarea['cell_area'].reindex(df_c.index)).T)
    df_prop    = add_meta((df_c[subcluster_cols].T / df_cellcount['cell_area'].reindex(df_c.index)).T)

    return df_count, df_density, df_prop, subcluster_cols


def run_statistics(df, metric_col, response_order):
    """Return dict of p-values for Pre↔Resection (paired) and cross-group comparisons."""
    stats = {}
    # Within-response: paired t-test Pre vs Resection
    for resp in response_order:
        sub = df[df['Response'] == resp].sort_values(['Patient', 'Timepoint'])
        pre = sub[sub['Timepoint'] == 'Pre'][metric_col].values
        res = sub[sub['Timepoint'] == 'Resection'][metric_col].values
        n = min(len(pre), len(res))
        if n >= 3:
            _, p = ttest_rel(pre[:n], res[:n])
            stats[f'{resp}_Pre_vs_Resection'] = p

    # Cross-group: Welch's t-test at Pre and Resection
    r0, r1, r2 = response_order
    for tp in ['Pre', 'Resection']:
        g = {r: df[(df['Timepoint'] == tp) & (df['Response'] == r)][metric_col] for r in response_order}
        _, stats[f'{tp}_{r0}_vs_{r1}'] = ttest_ind(g[r0], g[r1], equal_var=False, nan_policy='omit')
        _, stats[f'{tp}_{r0}_vs_{r2}'] = ttest_ind(g[r0], g[r2], equal_var=False, nan_policy='omit')
    return stats


def plot_temporal_dynamics(df, metric_col, response_order, title, output_path):
    """Three-panel longitudinal line plot (one panel per response group).

    Marker shape encodes DFS status: ● = DFS, ✕ = PD.
    Statistical summaries appear in suptitle.
    """
    stats = run_statistics(df, metric_col, response_order)

    fig = plt.figure(figsize=(9, 4.5))
    spec = gridspec.GridSpec(1, 3, width_ratios=[1, 1, 0.7], wspace=0.15)
    axes = [fig.add_subplot(spec[i]) for i in range(3)]
    for ax in axes[1:]:
        ax.sharey(axes[0])

    for i, resp in enumerate(response_order):
        info = RESPONSE_GROUPS[resp]
        sub = df[df['Response'] == resp].copy()

        for patient, dfs_status in zip(info['patients'], info['dfs']):
            pat_df = sub[sub['Patient'] == patient].sort_values('Timepoint')
            if pat_df.empty:
                continue
            valid = pat_df[metric_col].notna()
            axes[i].plot(
                pat_df.loc[valid, 'Timepoint'].astype(str).tolist(),
                pat_df.loc[valid, metric_col].tolist(),
                marker='o' if dfs_status == 'DFS' else 'x',
                color=info['color'],
                linewidth=1, markersize=5, alpha=0.55
            )

        axes[i].set_title(resp, fontweight='bold')
        axes[i].set_xticks(
            ['Pre', 'Resection'] if resp == 'pCR' else TIMEPOINT_ORDER
        )
        axes[i].tick_params(axis='x', rotation=30)
        axes[i].grid(axis='y', linestyle='--', alpha=0.4)
        axes[i].spines[['top', 'right']].set_visible(False)

        if i == 0:
            axes[i].set_ylabel(title)
            axes[i].spines['left'].set_visible(True)
        else:
            axes[i].spines['left'].set_visible(False)
            axes[i].tick_params(labelleft=False)

        # Within-response p-value
        key = f'{resp}_Pre_vs_Resection'
        if key in stats:
            axes[i].text(0.5, 0.93, f'Pre→Res p={stats[key]:.3f}',
                         transform=axes[i].transAxes, ha='center', fontsize=8)

    r0, r1, r2 = response_order
    sup_parts = []
    for tp in ['Pre', 'Resection']:
        k1 = f'{tp}_{r0}_vs_{r1}'
        k2 = f'{tp}_{r0}_vs_{r2}'
        if k1 in stats:
            sup_parts.append(f'{tp}: {r0} vs {r1} p={stats[k1]:.3f}, {r0} vs {r2} p={stats[k2]:.3f}')
    fig.suptitle(' | '.join(sup_parts), fontsize=8, y=1.01)

    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close(fig)


## 7. Tumour Subcluster Temporal Dynamics

In [ ]:
adata_tumor.obs['Timepoint'] = pd.Categorical(
    adata_tumor.obs['Sample'].str.split('_', 1).str[1],
    categories=TIMEPOINT_ORDER, ordered=True
)
adata_tumor.obs['Response'] = adata_tumor.obs['Sample'].apply(sample_to_response)
adata_tumor.obs['Patient']  = adata_tumor.obs['Sample'].str.split('_').str[0]

df_tc, df_td, df_tp, tumor_subclusters = build_metric_frames(
    adata_tumor, sample_name_ls, groupby='leiden'
)

for metric_label, df_m in [('count', df_tc), ('density', df_td), ('proportion', df_tp)]:
    for subcluster in tumor_subclusters:
        if subcluster not in df_m.columns:
            continue
        outpath = f'../figures/07_tumor_{subcluster.replace(" ", "_")}_{metric_label}.png'
        plot_temporal_dynamics(
            df_m, subcluster,
            response_order=['NR', 'MPR', 'pCR'],
            title=f'{subcluster} ({metric_label})',
            output_path=outpath
        )

# Export summary tables
df_tc.to_csv('../results/07_tumor_subcluster_count.csv')
df_td.to_csv('../results/07_tumor_subcluster_density.csv')
df_tp.to_csv('../results/07_tumor_subcluster_proportion.csv')
print('Tumour temporal dynamics complete.')


## 8. UMAP Embedding Density — Tumour Cells

`sc.tl.embedding_density` estimates a 2-D kernel density on the UMAP for each group defined by the combination of Timepoint and Response. This reveals whether specific tumour subtypes expand or contract following treatment.

In [ ]:
# Create combined Timepoint_Response key
adata_tumor.obs['Timepoint_Response'] = (
    adata_tumor.obs['Timepoint'].astype(str) + '_' +
    adata_tumor.obs['Response'].astype(str)
)
group_order = [
    'Pre_NR',    'JustAfter_NR',    'Resection_NR',
    'Pre_MPR',   'JustAfter_MPR',   'Resection_MPR',
    'Pre_pCR',                      'Resection_pCR',
]
# Keep only groups present in the data
group_order = [g for g in group_order if g in adata_tumor.obs['Timepoint_Response'].values]

adata_tumor.obs['Timepoint_Response'] = pd.Categorical(
    adata_tumor.obs['Timepoint_Response'], categories=group_order
)

sc.tl.embedding_density(adata_tumor, basis='umap', groupby='Timepoint_Response')
sc.pl.embedding_density(
    adata_tumor, basis='umap',
    key='umap_density_Timepoint_Response',
    ncols=3,
    save='_07_tumor_embedding_density.png'
)


## 9. T Cell Subcluster Temporal Dynamics

In [ ]:
adata_tcell = adata[adata.obs['Cell_type_integrate'] == 'T cell'].copy()

df_tc, df_td, df_tp, tcell_subclusters = build_metric_frames(
    adata_tcell, sample_name_ls, groupby='subcluster'
)

for metric_label, df_m in [('count', df_tc), ('density', df_td), ('proportion', df_tp)]:
    for subcluster in tcell_subclusters:
        if subcluster not in df_m.columns:
            continue
        outpath = f'../figures/07_tcell_{subcluster.replace(" ", "_").replace("/", "-")}_{metric_label}.png'
        plot_temporal_dynamics(
            df_m, subcluster,
            response_order=['NR', 'MPR', 'pCR'],
            title=f'{subcluster} ({metric_label})',
            output_path=outpath
        )

df_tc.to_csv('../results/07_tcell_subcluster_count.csv')
df_td.to_csv('../results/07_tcell_subcluster_density.csv')
df_tp.to_csv('../results/07_tcell_subcluster_proportion.csv')
print('T cell temporal dynamics complete.')


## 10. Myeloid Subcluster Temporal Dynamics

In [ ]:
adata_mye = adata[adata.obs['Cell_type_integrate'] == 'Myeloid'].copy()

df_tc, df_td, df_tp, mye_subclusters = build_metric_frames(
    adata_mye, sample_name_ls, groupby='subcluster'
)

for metric_label, df_m in [('count', df_tc), ('density', df_td), ('proportion', df_tp)]:
    for subcluster in mye_subclusters:
        if subcluster not in df_m.columns:
            continue
        outpath = f'../figures/07_myeloid_{subcluster.replace(" ", "_").replace("/", "-")}_{metric_label}.png'
        plot_temporal_dynamics(
            df_m, subcluster,
            response_order=['NR', 'MPR', 'pCR'],
            title=f'{subcluster} ({metric_label})',
            output_path=outpath
        )

df_tc.to_csv('../results/07_myeloid_subcluster_count.csv')
df_td.to_csv('../results/07_myeloid_subcluster_density.csv')
df_tp.to_csv('../results/07_myeloid_subcluster_proportion.csv')
print('Myeloid temporal dynamics complete.')


## 11. Stromal/CAF Subcluster Temporal Dynamics

In [ ]:
adata_caf = adata[adata.obs['Cell_type_integrate'].isin(['CAF', 'Stromal', 'Fibroblast', 'Endothelial'])].copy()

df_tc, df_td, df_tp, caf_subclusters = build_metric_frames(
    adata_caf, sample_name_ls, groupby='subcluster'
)

for metric_label, df_m in [('count', df_tc), ('density', df_td), ('proportion', df_tp)]:
    for subcluster in caf_subclusters:
        if subcluster not in df_m.columns:
            continue
        outpath = f'../figures/07_stromal_{subcluster.replace(" ", "_").replace("/", "-")}_{metric_label}.png'
        plot_temporal_dynamics(
            df_m, subcluster,
            response_order=['NR', 'MPR', 'pCR'],
            title=f'{subcluster} ({metric_label})',
            output_path=outpath
        )

df_tc.to_csv('../results/07_stromal_subcluster_count.csv')
df_td.to_csv('../results/07_stromal_subcluster_density.csv')
df_tp.to_csv('../results/07_stromal_subcluster_proportion.csv')
print('Stromal/CAF temporal dynamics complete.')


## 12. Summary Statistics Table

Collect all pairwise statistical test results into a single CSV for downstream reporting.

In [ ]:
def collect_stats_table(df, subclusters, label, response_order=['NR', 'MPR', 'pCR']):
    """Return a DataFrame of p-values for all subcluster × comparison combinations."""
    rows = []
    for sc_name in subclusters:
        if sc_name not in df.columns:
            continue
        stats = run_statistics(df, sc_name, response_order)
        row = {'cell_type_label': label, 'subcluster': sc_name}
        row.update(stats)
        rows.append(row)
    return pd.DataFrame(rows)

# Re-build frames for each cell type (already done above; call again if needed)
stat_frames = []
for label, adata_sub, cols in [
    ('Tumour',  adata_tumor, tumor_subclusters),
    ('T cell',  adata_tcell, tcell_subclusters),
    ('Myeloid', adata_mye,   mye_subclusters),
    ('Stromal', adata_caf,   caf_subclusters),
]:
    _, df_density, _, _ = build_metric_frames(adata_sub, sample_name_ls, groupby='leiden' if label == 'Tumour' else 'subcluster')
    stat_frames.append(collect_stats_table(df_density, cols, label))

df_stats_all = pd.concat(stat_frames, ignore_index=True)
df_stats_all.to_csv('../results/07_temporal_statistics_density.csv', index=False)
print('Summary statistics saved.')
print(df_stats_all.head(10))
